# PIPELINE TIỀN XỬ LÝ DỮ LIỆU DỰ BÁO PHỤ TẢI ĐIỆN 30 PHÚT
**Đề tài:** Nghiên cứu mô hình học sâu dự báo phụ tải điện trạm Điện lực Tây Ninh  
**Mục tiêu:** Xử lý chuỗi thời gian đo đếm theo chu kỳ 30 phút, lọc nhiễu, điền khuyết có kiểm soát, trích xuất đặc trưng và chuẩn bị dữ liệu huấn luyện học có giám sát.

## 1. Cấu hình môi trường và thư viện

In [ ]:
import os
import sys
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình font và hiển thị đồ thị
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# Tự động nhận diện đường dẫn (Google Colab hoặc Máy cục bộ)
if Path("/content/drive/MyDrive/Tam_TLU/datasets").exists():
    DATA_DIR = Path("/content/drive/MyDrive/Tam_TLU/datasets")
elif Path("/content/drive/MyDrive/power-load-forecasting/Tam_TLU/datasets").exists():
    DATA_DIR = Path("/content/drive/MyDrive/power-load-forecasting/Tam_TLU/datasets")
elif Path("../datasets").exists():
    DATA_DIR = Path("../datasets")
else:
    DATA_DIR = Path("./datasets")

OUTPUT_DIR = Path("./output_preprocessing")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUMERIC_COLS = ["UA", "UB", "UC", "IA", "IB", "IC", "CosF", "P_kW", "Q_kVAr"]

print(f"Thư mục dữ liệu: {DATA_DIR.resolve()}")
print(f"Thư mục xuất kết quả: {OUTPUT_DIR.resolve()}")

## 2. Quét và tải toàn bộ các file CSV (Bỏ qua file nén zip/7z)

In [ ]:
# Tìm toàn bộ file CSV đệ quy
all_csv_paths = sorted(DATA_DIR.rglob("*.csv"))
exclude_kw = ["report", "summary", "cleaned", "featured", "supervised", "train", "validation", "test"]
valid_csvs = [p for p in all_csv_paths if not any(kw in p.name.lower() for kw in exclude_kw)]

print(f"Đã tìm thấy {len(valid_csvs)} file CSV đo đếm hợp lệ.")
for p in valid_csvs[:5]:
    print(f"  - Mẫu file: {p.relative_to(DATA_DIR)}")

## 3. Chuẩn hóa tên cột, kiểu dữ liệu thời gian và số liệu

In [ ]:
def parse_time_robust(df):
    time_col = next((c for c in ["TIME", "Time", "ThoiGian", "THOIGIAN"] if c in df.columns), None)
    if not time_col:
        raise ValueError(f"Thiếu cột thời gian: {df.columns.tolist()}")
    s = df[time_col].astype(str).str.strip()
    parsed = pd.to_datetime(s, errors="coerce")
    if parsed.notna().mean() >= 0.8:
        return parsed
    date_col = next((c for c in ["DATE", "Date", "Ngay", "NGAY"] if c in df.columns), None)
    if date_col:
        s_date = pd.to_datetime(df[date_col], errors="coerce")
        s_time_fixed = s.str.replace(r"\s+", "", regex=True)
        s_time_fixed = s_time_fixed.where(s_time_fixed.str.count(":") == 2, "00:" + s_time_fixed)
        return s_date + pd.to_timedelta(s_time_fixed, errors="coerce")
    return parsed

dfs = []
for p in valid_csvs:
    try:
        df_i = pd.read_csv(p, low_memory=False, encoding="utf-8")
    except UnicodeDecodeError:
        df_i = pd.read_csv(p, low_memory=False, encoding="latin1")
    
    df_i.columns = [str(c).strip() for c in df_i.columns]
    df_i["TIME"] = parse_time_robust(df_i)
    df_i = df_i.dropna(subset=["TIME"]).copy()
    
    for c in NUMERIC_COLS:
        if c in df_i.columns:
            s_num = df_i[c].astype(str).str.strip().str.replace(",", ".", regex=False)
            df_i[c] = pd.to_numeric(s_num, errors="coerce")
            
    if "MaDiemDo" in df_i.columns:
        df_i["MaDiemDo"] = pd.to_numeric(df_i["MaDiemDo"], errors="coerce")
        df_i = df_i.dropna(subset=["MaDiemDo"]).copy()
        df_i["MaDiemDo"] = df_i["MaDiemDo"].astype(int)
    if "TenDiemDo" in df_i.columns:
        df_i["TenDiemDo"] = df_i["TenDiemDo"].astype(str).str.strip()
        
    df_i["source_file"] = p.name
    dfs.append(df_i)

df_raw = pd.concat(dfs, ignore_index=True)
print(f"Tổng số dòng dữ liệu thô: {len(df_raw):,}")

## 4. Kiểm tra chất lượng dữ liệu & Xuất `quality_report.csv`

In [ ]:
total_rows = len(df_raw)
unique_points = df_raw["MaDiemDo"].nunique() if "MaDiemDo" in df_raw.columns else 0
dup_count = int(df_raw.duplicated(subset=["TIME", "MaDiemDo"]).sum())

missing_stats = df_raw.isna().sum()
missing_pct = (missing_stats / total_rows * 100).round(2)

report_rows = []
for c in df_raw.columns:
    report_rows.append({
        "column_name": c,
        "data_type": str(df_raw[c].dtype),
        "missing_count": int(missing_stats[c]),
        "missing_pct": float(missing_pct[c]),
        "unique_values": int(df_raw[c].nunique())
    })

df_quality_report = pd.DataFrame(report_rows)
df_quality_report.to_csv(OUTPUT_DIR / "quality_report.csv", index=False, encoding="utf-8-sig")

print(f"- Số điểm đo: {unique_points}")
print(f"- Dải thời gian: {df_raw['TIME'].min()} -> {df_raw['TIME'].max()}")
print(f"- Số dòng trùng lặp: {dup_count:,}")
print("Đã xuất: quality_report.csv")

## 5. Quy chuẩn lưới thời gian 30 phút & Nội suy có kiểm soát (Tối đa 2 bước)

In [ ]:
# Làm tròn thời gian về mốc 30 phút gần nhất (00:00, 00:30,...)
df_raw["TIME_30m"] = df_raw["TIME"].dt.round("30min")
df_sorted = df_raw.sort_values(by=["MaDiemDo", "TIME_30m", "TIME"]).reset_index(drop=True)
df_dedup = df_sorted.drop_duplicates(subset=["MaDiemDo", "TIME_30m"], keep="first").copy()

resampled_list = []
for point_id, group in df_dedup.groupby("MaDiemDo"):
    group = group.set_index("TIME_30m").sort_index()
    timeline = pd.date_range(start=group.index.min(), end=group.index.max(), freq="30min", name="TIME")
    
    reindexed = group.reindex(timeline)
    reindexed["MaDiemDo"] = point_id
    if "TenDiemDo" in group.columns:
        reindexed["TenDiemDo"] = group["TenDiemDo"].dropna().iloc[0] if len(group["TenDiemDo"].dropna()) > 0 else f"Point_{point_id}"
    if "MBA" in group.columns:
        reindexed["MBA"] = group["MBA"].dropna().iloc[0] if len(group["MBA"].dropna()) > 0 else "Unknown"
        
    was_missing = reindexed["P_kW"].isna()
    present_nums = [c for c in NUMERIC_COLS if c in reindexed.columns]
    
    # Chỉ nội suy tối đa 2 bước thời gian liên tiếp (1 tiếng)
    imputed = reindexed[present_nums].interpolate(method="time", limit=2, limit_direction="forward")
    reindexed["is_imputed"] = (was_missing & imputed["P_kW"].notna()).astype(int)
    reindexed[present_nums] = imputed
    
    resampled_list.append(reindexed.reset_index().rename(columns={"index": "TIME"}))

df_clean_long = pd.concat(resampled_list, ignore_index=True)
df_clean_long.to_csv(OUTPUT_DIR / "cleaned_long.csv", index=False, encoding="utf-8-sig")

# Chuyển sang bảng ngang (cleaned_wide.csv)
df_wide = df_clean_long.pivot_table(index="TIME", columns="MaDiemDo", values=["P_kW", "Q_kVAr"], aggfunc="first")
df_wide.columns = [f"{var}_{pt}" for (var, pt) in df_wide.columns]
df_wide.reset_index().to_csv(OUTPUT_DIR / "cleaned_wide.csv", index=False, encoding="utf-8-sig")

print(f"Đã xuất cleaned_long.csv ({len(df_clean_long):,} dòng) và cleaned_wide.csv")

## 6. Trích xuất đặc trưng vật lý & chuỗi thời gian (Feature Engineering)

In [ ]:
df_feat = df_clean_long.copy()
eps = 1e-6

# Đặc trưng 3 pha
if set(["UA", "UB", "UC"]).issubset(df_feat.columns):
    df_feat["U_avg"] = df_feat[["UA", "UB", "UC"]].mean(axis=1)
    df_feat["U_max"] = df_feat[["UA", "UB", "UC"]].max(axis=1)
    df_feat["U_min"] = df_feat[["UA", "UB", "UC"]].min(axis=1)
    df_feat["U_unbalance_pct"] = (df_feat["U_max"] - df_feat["U_min"]) / (df_feat["U_avg"].abs() + eps) * 100

if set(["IA", "IB", "IC"]).issubset(df_feat.columns):
    df_feat["I_avg"] = df_feat[["IA", "IB", "IC"]].mean(axis=1)
    df_feat["I_max"] = df_feat[["IA", "IB", "IC"]].max(axis=1)
    df_feat["I_min"] = df_feat[["IA", "IB", "IC"]].min(axis=1)
    df_feat["I_unbalance_pct"] = (df_feat["I_max"] - df_feat["I_min"]) / (df_feat["I_avg"].abs() + eps) * 100

# Công suất biểu kiến S (kVA)
df_feat["S_kVA"] = np.sqrt(df_feat["P_kW"].fillna(0)**2 + df_feat["Q_kVAr"].fillna(0)**2)

# Đặc trưng thời gian
df_feat["hour"] = df_feat["TIME"].dt.hour + df_feat["TIME"].dt.minute / 60.0
df_feat["dayofweek"] = df_feat["TIME"].dt.dayofweek
df_feat["dayofmonth"] = df_feat["TIME"].dt.day
df_feat["month"] = df_feat["TIME"].dt.month
df_feat["is_weekend"] = (df_feat["dayofweek"] >= 5).astype(int)
df_feat["sin_hour"] = np.sin(2 * np.pi * df_feat["hour"] / 24.0)
df_feat["cos_hour"] = np.cos(2 * np.pi * df_feat["hour"] / 24.0)

# Tạo lags và rolling thống kê theo từng MaDiemDo (Shift 1 để tránh rò rỉ)
feat_groups = []
for point_id, grp in df_feat.groupby("MaDiemDo"):
    grp = grp.sort_values("TIME").copy()
    for lag in [1, 2, 3, 48, 336]:
        grp[f"lag_{lag}"] = grp["P_kW"].shift(lag)
    p_prev = grp["P_kW"].shift(1)
    grp["rolling_mean_3"] = p_prev.rolling(3, min_periods=1).mean()
    grp["rolling_mean_6"] = p_prev.rolling(6, min_periods=1).mean()
    grp["rolling_mean_48"] = p_prev.rolling(48, min_periods=1).mean()
    feat_groups.append(grp)

df_featured = pd.concat(feat_groups, ignore_index=True)
df_featured.to_csv(OUTPUT_DIR / "featured_data.csv", index=False, encoding="utf-8-sig")
print("Đã xuất: featured_data.csv")

## 7. Xác định trạm lớn & Bảng tổng hợp `station_summary.csv`

In [ ]:
station_rows = []
for pid, grp in df_clean_long.groupby("MaDiemDo"):
    name = grp["TenDiemDo"].iloc[0] if "TenDiemDo" in grp.columns else f"Point_{pid}"
    station_rows.append({
        "MaDiemDo": pid,
        "TenDiemDo": name,
        "total_records": len(grp),
        "P_mean_kW": round(float(grp["P_kW"].mean()), 3),
        "P_max_kW": round(float(grp["P_kW"].max()), 3),
        "missing_rate_pct": round(float(grp["P_kW"].isna().mean() * 100), 2)
    })

df_station_summary = pd.DataFrame(station_rows).sort_values(by="P_mean_kW", ascending=False).reset_index(drop=True)
df_station_summary.to_csv(OUTPUT_DIR / "station_summary.csv", index=False, encoding="utf-8-sig")
display(df_station_summary)

# Chọn điểm đo có công suất lớn nhất làm TARGET_POINT
TARGET_POINT = int(df_station_summary.iloc[0]["MaDiemDo"])
print(f"Trạm mục tiêu được chọn: MaDiemDo = {TARGET_POINT} ({df_station_summary.iloc[0]['TenDiemDo']})")

## 8. Thiết lập bài toán học có giám sát (Dự đoán sau 30 phút)

In [ ]:
df_target_subset = df_featured[df_featured["MaDiemDo"] == TARGET_POINT].sort_values("TIME").copy()

# Nhãn mục tiêu sau 30 phút: P[t+1]
df_target_subset["target_P_kW"] = df_target_subset["P_kW"].shift(-1)
df_supervised = df_target_subset.dropna(subset=["target_P_kW"]).copy()

# Mức biến thiên định lượng và nhãn xu hướng
df_supervised["target_delta_P"] = df_supervised["target_P_kW"] - df_supervised["P_kW"]
threshold_kw = 0.5
conditions = [
    df_supervised["target_delta_P"] > threshold_kw,
    df_supervised["target_delta_P"] < -threshold_kw
]
df_supervised["target_direction"] = np.select(conditions, ["TĂNG", "GIẢM"], default="KHÔNG ĐỔI")

df_supervised.to_csv(OUTPUT_DIR / "supervised_data.csv", index=False, encoding="utf-8-sig")
print(f"Đã xuất: supervised_data.csv ({len(df_supervised):,} mẫu)")

## 9. Phân chia tập Train (70%), Validation (15%), Test (15%) theo thời gian

In [ ]:
n = len(df_supervised)
train_idx = int(n * 0.70)
val_idx = int(n * 0.85)

train_df = df_supervised.iloc[:train_idx].copy()
val_df = df_supervised.iloc[train_idx:val_idx].copy()
test_df = df_supervised.iloc[val_idx:].copy()

train_df.to_csv(OUTPUT_DIR / "train.csv", index=False, encoding="utf-8-sig")
val_df.to_csv(OUTPUT_DIR / "validation.csv", index=False, encoding="utf-8-sig")
test_df.to_csv(OUTPUT_DIR / "test.csv", index=False, encoding="utf-8-sig")

print(f"- Train:      {len(train_df):,} mẫu ({train_df['TIME'].min()} -> {train_df['TIME'].max()})")
print(f"- Validation: {len(val_df):,} mẫu ({val_df['TIME'].min()} -> {val_df['TIME'].max()})")
print(f"- Test:       {len(test_df):,} mẫu ({test_df['TIME'].min()} -> {test_df['TIME'].max()})")

## 10. Tự động vẽ và lưu 6 biểu đồ kiểm tra chất lượng

In [ ]:
# 1. Công suất theo thời gian
fig, ax = plt.subplots(figsize=(14, 5))
sample_sub = df_target_subset.iloc[:1440]
ax.plot(sample_sub["TIME"], sample_sub["P_kW"], color="tab:blue", lw=1.2)
ax.set_title("1. Công suất P_kW theo thời gian (Mẫu 30 ngày)", fontsize=14, fontweight="bold")
fig.savefig(OUTPUT_DIR / "01_power_over_time.png", dpi=150, bbox_inches="tight")
plt.show()

# 2. Tỷ lệ thiếu dữ liệu
fig, ax = plt.subplots(figsize=(10, 4))
missing_p = (df_raw[NUMERIC_COLS].isna().mean() * 100).sort_values(ascending=False)
sns.barplot(x=missing_p.values, y=missing_p.index, ax=ax, palette="Blues_r")
ax.set_title("2. Tỷ lệ thiếu dữ liệu (%)", fontsize=14, fontweight="bold")
fig.savefig(OUTPUT_DIR / "02_missing_rate.png", dpi=150, bbox_inches="tight")
plt.show()

# 3. Phân phối P_kW
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df_clean_long["P_kW"].dropna(), bins=50, kde=True, ax=axes[0], color="tab:green")
axes[0].set_title("Histogram P_kW", fontsize=12, fontweight="bold")
sns.boxplot(y=df_clean_long["P_kW"].dropna(), ax=axes[1], color="lightgreen")
axes[1].set_title("Boxplot P_kW", fontsize=12, fontweight="bold")
fig.savefig(OUTPUT_DIR / "03_power_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

# 4. Đối chiếu trước và sau nội suy
fig, ax = plt.subplots(figsize=(14, 5))
raw_p = df_raw[df_raw["MaDiemDo"] == TARGET_POINT].set_index("TIME").sort_index()
clean_p = df_clean_long[df_clean_long["MaDiemDo"] == TARGET_POINT].set_index("TIME").sort_index()
t0, t1 = clean_p.index.min(), clean_p.index.min() + pd.Timedelta(days=7)
ax.plot(raw_p.loc[t0:t1].index, raw_p.loc[t0:t1, "P_kW"], "r.", label="Gốc")
ax.plot(clean_p.loc[t0:t1].index, clean_p.loc[t0:t1, "P_kW"], "b-", lw=1.2, label="Đã xử lý")
ax.set_title("4. So sánh dữ liệu gốc và sau nội suy (1 tuần)", fontsize=14, fontweight="bold")
ax.legend()
fig.savefig(OUTPUT_DIR / "04_imputation_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# 5. Phụ tải theo giờ
fig, ax = plt.subplots(figsize=(12, 4))
avg_h = df_clean_long.groupby(df_clean_long["TIME"].dt.hour)["P_kW"].mean()
ax.plot(avg_h.index, avg_h.values, marker="o", color="orange", lw=2)
ax.set_title("5. Phụ tải trung bình theo 24 giờ trong ngày", fontsize=14, fontweight="bold")
fig.savefig(OUTPUT_DIR / "05_avg_load_by_hour.png", dpi=150, bbox_inches="tight")
plt.show()

# 6. Phụ tải theo ngày trong tuần
fig, ax = plt.subplots(figsize=(10, 4))
days = ["T2", "T3", "T4", "T5", "T6", "T7", "CN"]
avg_d = df_clean_long.groupby(df_clean_long["TIME"].dt.dayofweek)["P_kW"].mean()
sns.barplot(x=[days[i] for i in avg_d.index], y=avg_d.values, ax=ax, palette="Purples_d")
ax.set_title("6. Phụ tải trung bình theo ngày trong tuần", fontsize=14, fontweight="bold")
fig.savefig(OUTPUT_DIR / "06_avg_load_by_weekday.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Báo cáo tổng kết và Lưu metadata `preprocessing_summary.json`

In [ ]:
summary = {
    "created_at": datetime.now().isoformat(),
    "csv_files_count": len(valid_csvs),
    "raw_rows": len(df_raw),
    "clean_rows": len(df_clean_long),
    "unique_points": int(df_clean_long["MaDiemDo"].nunique()),
    "target_point": TARGET_POINT,
    "train_samples": len(train_df),
    "val_samples": len(val_df),
    "test_samples": len(test_df),
    "data_ready_for_training": True
}

with open(OUTPUT_DIR / "preprocessing_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4, ensure_ascii=False)

print("=" * 60)
print("TỔNG KẾT PIPELINE TIỀN XỬ LÝ DỮ LIỆU PHỤ TẢI 30 PHÚT")
print("=" * 60)
print(f"- Số file CSV đã đọc:      {summary['csv_files_count']}")
print(f"- Số dòng dữ liệu thô:     {summary['raw_rows']:,}")
print(f"- Số dòng sau làm sạch:    {summary['clean_rows']:,}")
print(f"- Trạm lớn mục tiêu:       MaDiemDo = {TARGET_POINT}")
print(f"- Số mẫu Train / Val / Test: {summary['train_samples']:,} / {summary['val_samples']:,} / {summary['test_samples']:,}")
print(f"- Đủ điều kiện train model: {summary['data_ready_for_training']}")
print("=" * 60)